Leetcode #350

Easy

Given two integer arrays nums1 and num2, return an array of their intersection. Each element in the result must appear as many times as it shows up in both arrays and you may return the result in any order. 



Example 1

text
Input: nums1 =, nums2 = [2, 2]
Output: [2, 2]
Use code with caution.

Example 2

text
Input: nums1 = [4,9,5], nums2 = [9,4,9,8,4]
Output: [4, 9] (or)
Explanation: [9, 4, 9, 4] is incorrect because '9' only appears once in nums1, and '4' only appears once in nums2.
Use code with caution.

Key Constraint Rule

You must respect multiplicity/frequency. If the number 2 appears 4 times in nums1 and 2 times in nums2, it must appear exactly min(4, 2) = 2 times in your final output array.

In [ ]:
def intersect_naive(nums1, nums2):
    result = []
    
    # Copy nums2 to avoid modifying the original input array parameters
    nums2_copy = list(nums2)
    
    # Loop through every element in the first array
    for num1 in nums1:
        # Check if the element exists in our copy of the second array
        if num1 in nums2_copy:
            result.append(num1)
            # Remove ONLY the first occurrence of this element to handle duplicates correctly
            nums2_copy.remove(num1)
            
    return result

# --- Verification ---
n1 = [4, 9, 5]
n2 = [9, 4, 9, 8, 4]
print(intersect_naive(n1, n2))  
# Output: [4, 9] (Matches the multiplicity rule perfectly)



Complexity Analysis of the Naive Approach

• Time Complexity: O(N * M), where N is the length of nums1 and M is the length of nums2.
	• For each element in nums1, Python performs a linear scan (if num1 in nums2_copy) which takes O(M) time.
	• If a match is found, .remove(num1) performs another linear scan to find and delete the item, shifting all subsequent elements down in memory, taking O(M) time.
• Space Complexity: O(M) to create the helper storage duplicate list nums2_copy.

Why the Hash Map Approach is Better

The optimal Hash Map (Counter) approach handles this in O(N + M) linear time because dictionary lookups and decrements occur instantly in O(1) time, completely eliminating the slow element-shifting behavior of .remove().

Since we need to keep track of counts instead of just presence, a single Hash Set will no longer work. We need a Hash Map (Frequency Map).

The Optimal Hash Map Strategy

1. Scan through nums1 and count how many times each number appears using a dictionary.
2. Scan through nums2. For every number you encounter, check if it exists in your dictionary with a count greater than 0.
3. If it does, add that number to your results list and decrement its count by 1 in the dictionary.


In [ ]:

from collections import Counter

def intersect(nums1, nums2):
    # 1. Map out the frequencies of the first array
    counts = Counter(nums1)
    result = []
    
    # 2. Match against the second array
    for num in nums2:
        if counts[num] > 0:
            result.append(num)
            counts[num] -= 1  # Reduce the available count pool
            
    return result

# Usage
nums1 = [4,9,5]
nums2 = [9,4,9,8,4]

result = intersect(nums1, nums2)
print (result)

nums1 = [4, 9, 9, 5]
nums2 = [9,4,9,8,4]

result = intersect(nums1, nums2)
print (result)

[9, 4]
[9, 4, 9]




Complexity Analysis

• Time Complexity: O(N + M) because we loop through nums1 once to count frequencies and nums2 once to check matches.
• Space Complexity: O(min(N, M)) to store the counts of the smaller array inside our dictionary.

Interviewers love asking three follow-up questions for LeetCode 350 to check your optimization skills:
1. What if the given arrays are already sorted? How would you optimize it?

2. What if nums1's size is small compared to nums2's size? Which algorithm is better?

3. What if elements of nums2 are stored on disk and the memory is limited such that you cannot load all elements into the RAM at once?



Question 1: What if the given arrays are already sorted? How would you optimize it?

If both arrays are already sorted, you can completely eliminate the Hash Map and achieve O(1) constant auxiliary space by using a Two-Pointer technique.

The Strategy

1. Place pointer i at the start of nums1 and pointer j at the start of nums2.
2. Compare the elements:
	• If nums1[i] == nums2[j], you found a match! Add it to the results and advance both pointers (i++, j++).
	• If nums1[i] < nums2[j], then nums1[i] is too small to ever match anything further down in nums2. Advance i.
	• If nums1[i] > nums2[j], advance j.


In [ ]:
def intersect_sorted(nums1, nums2):
    # Assumes arrays are already sorted
    i, j = 0, 0
    result = []
    
    while i < len(nums1) and j < len(nums2):
        if nums1[i] == nums2[j]:
            result.append(nums1[i])
            i += 1
            j += 1
        elif nums1[i] < nums2[j]:
            i += 1
        else:
            j += 1
            
    return result


• Time Complexity: O(m + n) — A clean single-pass sweep.
• Space Complexity: O(1) extra space — No extra data structures needed.


Question 2: What if nums1's size is small compared to nums2s size? Which algorithm is better?

If one array is significantly smaller than the other (e.g., nums1 has 10 elements and nums2 has 10 million elements), the standard Hash Map approach is best, but you must map the smaller array.

• Why? Mapping nums1 means your frequency dictionary only consumes memory for 10 entries instead of 10 million. 
Your space complexity drops to O(min(m, n)).

• Alternative (If sorted): If the arrays are already sorted and one is tiny, you can loop through the tiny array and perform a Binary Search inside the massive array for each element. 

This takes O(m log n) time, which is highly efficient when (m << n).



Question 3: What if elements of nums2 are stored on disk and the memory is limited such that you cannot load all elements into the RAM at once?

This is a classic External Memory / Big Data scenario. You cannot fit everything into memory, so you must stream the data in chunks (blocks).

Scenario A: nums1 fits in RAM, but nums2 does not

1. Load all of nums1 into a Hash Map frequency tracker in memory (since it fits).
2. Open a streaming file pointer to nums2 on disk.
3. Read nums2 sequentially, one block/chunk at a time (e.g., 1,000 numbers at a time).
4. Process each chunk against your in-memory Hash Map, decrementing counts and outputting matches immediately to an output stream.

Scenario B: Neither array fits in RAM

If both arrays are completely massive, you must use External Merge Sort:
1. Sort pieces of both files individually on disk and write out smaller sorted sub-files.
2. Use a multi-way merge stream to read the heads of the sorted sub-files simultaneously into RAM.
3. Apply the Two-Pointer technique (from Question 1) as you stream the sorted items from both collections past each other, outputting intersections on the fly. This forms the foundational logic behind database operations like Hash Joins and Sort-Merge Joins in SQL servers.
